# Capture and share desktop audio

This notebook captures one running desktop application. The microphone, recording, and Relay publishing are off until you enable them in the configuration cell. Each selected source remains a separate Stem with its own identity, timing, delivery observations, and recording.

The notebook uses the installed `pocketstation` package. It does not load code from a repository checkout.

In [ ]:
import { connector, Session, Source } from "pocketstation/node";

const automatedApplication = Deno.env.get("POCKETSTATION_APPLICATION");
const applicationValue = automatedApplication ?? prompt("Application to capture");
if (applicationValue === null || applicationValue.trim().length === 0) {
  throw new Error("Enter the exact name, application ID, or process ID of a running application");
}

const application = /^[1-9][0-9]*$/.test(applicationValue)
  ? Number(applicationValue)
  : applicationValue;
const microphoneRequested = /^(1|true|yes)$/i.test(
  Deno.env.get("POCKETSTATION_MICROPHONE") ?? "false",
);
const microphoneDeviceId = Deno.env.get("POCKETSTATION_MICROPHONE_DEVICE_ID");
const recordingRoot = Deno.env.get("POCKETSTATION_RECORDING_ROOT");
const relayUrl = Deno.env.get("POCKETSTATION_RELAY_URL");
const relaySessionId = Deno.env.get("POCKETSTATION_RELAY_SESSION_ID");
const relaySourceToken = Deno.env.get("POCKETSTATION_RELAY_SOURCE_TOKEN");
const runForMs = Number(Deno.env.get("POCKETSTATION_RUN_FOR_MS") ?? "15000");
if (!Number.isInteger(runForMs) || runForMs < 1000 || runForMs > 120000) {
  throw new Error("POCKETSTATION_RUN_FOR_MS must be an integer from 1000 through 120000");
}
if ([relayUrl, relaySessionId, relaySourceToken].filter(Boolean).length === 1 ||
    [relayUrl, relaySessionId, relaySourceToken].filter(Boolean).length === 2) {
  throw new Error("Set all three Relay values or leave all three unset");
}

({
  application: applicationValue,
  microphone: microphoneRequested ? (microphoneDeviceId ?? "default") : false,
  recording: recordingRoot ?? false,
  relay: relayUrl !== undefined,
  runForMs,
});

## Compose one Session

The application and optional microphone each fan out from one capture operation. The Connector counts frames without retaining audio. Relay and recording are added only when their configuration is present.

In [ ]:
const framesBySource = new Map();
const observation = connector((frame) => {
  const sourceId = frame.sourceId.toString();
  framesBySource.set(sourceId, (framesBySource.get(sourceId) ?? 0) + 1);
}, { capacityFrames: 32 });

const session = new Session({
  channels: 1,
  frameDurationMs: 20,
  ...(recordingRoot === undefined ? {} : { recordingRoot }),
});
const applicationStem = session.capture(Source.application(application));
const microphoneStem = microphoneRequested
  ? session.capture(Source.microphone(microphoneDeviceId))
  : undefined;

applicationStem.sendTo(observation);
microphoneStem?.sendTo(observation);
if (recordingRoot !== undefined) {
  applicationStem.record("application");
  microphoneStem?.record("microphone");
}
if (relayUrl !== undefined && relaySessionId !== undefined && relaySourceToken !== undefined) {
  const relay = session.relay({
    url: relayUrl,
    sessionId: relaySessionId,
    sourceToken: relaySourceToken,
    startupTimeoutMs: 20000,
  });
  applicationStem.send(relay.audio("application"));
  microphoneStem?.send(relay.audio("microphone"));
}

({
  applicationStemId: applicationStem.id.toString(),
  microphoneStemId: microphoneStem?.id.toString(),
});

## Run and inspect the result

Starting waits for every declared destination to become ready. Stopping finishes accepted work, closes the sources and Connector, and finalizes each requested recording.

In [ ]:
const readyFile = Deno.env.get("POCKETSTATION_READY_FILE");
const stopFile = Deno.env.get("POCKETSTATION_STOP_FILE");
const resultFile = Deno.env.get("POCKETSTATION_RESULT_FILE");
const json = (_key, value) => typeof value === "bigint" ? value.toString() : value;
const sleep = (milliseconds) => new Promise((resolve) => setTimeout(resolve, milliseconds));

const running = await session.start();
let outcome;
try {
  if (readyFile !== undefined) {
    await Deno.writeTextFile(readyFile, JSON.stringify({
      status: "ready",
      nativeSessionId: running.sessionId.toString(),
    }) + "\n");
  }
  const deadline = Date.now() + runForMs;
  while (Date.now() < deadline) {
    if (stopFile !== undefined) {
      try {
        await Deno.stat(stopFile);
        break;
      } catch (failure) {
        if (!(failure instanceof Deno.errors.NotFound)) throw failure;
      }
    }
    await sleep(200);
  }
  const liveMetrics = await running.metrics();
  const stopResult = await running.stop();
  outcome = {
    ...stopResult,
    status: "passed",
    sourceFrames: Object.fromEntries(framesBySource),
    liveMetrics,
  };
} catch (failure) {
  const cleanup = await running.cancel();
  outcome = { status: "failed", error: String(failure), cleanup };
  throw failure;
} finally {
  if (resultFile !== undefined && outcome !== undefined) {
    await Deno.writeTextFile(resultFile, JSON.stringify(outcome, json, 2) + "\n");
  }
}

Deno.jupyter.md`## Session complete

- Sources observed: **${Object.keys(outcome.sourceFrames).length}**
- Frames delivered to the example Connector: **${Object.values(outcome.sourceFrames).reduce((sum, count) => sum + count, 0)}**
- Recording: **${outcome.recording?.state ?? "not requested"}**
- Session: **${outcome.sessionState}**`;